# ?? 

## xgboost

- https://apple-rbox.tistory.com/6
- https://brunch.co.kr/@snobberys/137


## lightgbm

- https://ko.raw3h.net/page/what-is-lightgbm-how-to-implement-it-how-to-fine-tune-the-parameters-5295f7/


## ? ?

- https://www.kaggle.com/shep312/applying-lightgbm-to-titanic-dataset
- https://www.kaggle.com/suniliitb96/titanic-survival-prediction-using-xgboost


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import plot_importance
from xgboost import XGBClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

In [ ]:
def metrics(y_test, pred):
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    roc_score = roc_auc_score(y_test, pred)
    print('??? : {0:.2f}, ??? : {1:.2f}, ??? : {2:.2f}'.format(accuracy, precision, recall))
    print('f1-score : {0:.2f}, auc : {1:.2f}'.format(f1, roc_score))

In [ ]:
data = load_breast_cancer()

cancer = pd.DataFrame(data.data, columns = data.feature_names)
cancer['target'] = data.target
cancer.head()

In [ ]:
cancer['target'].value_counts()

In [ ]:
y = cancer['target']
X = cancer.drop('target', axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 11)
print(X_train.shape, X_test.shape)

In [ ]:
xgb = XGBClassifier(n_estimators=500, learning_rate = 0.1, max_depth = 4)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

In [ ]:
metrics(y_test, xgb_pred)

In [ ]:
data = pd.read_csv('./datas/titanic/train.csv')
data.head()

In [ ]:
data['Embarked'].fillna('S', inplace = True)
data['Fare'].fillna(0, inplace=True)
data['Initial'] = data['Name'].str.extract('([A-Za-z]+)\.')
data['Initial'].replace(['Mlle','Mme','Ms','Dr','Major','Lady','Countess','Jonkheer','Col','Rev','Capt','Sir','Don', 'Dona'],['Miss','Miss','Miss','Mr','Mr','Mrs','Mrs','Other','Other','Other','Mr','Mr','Mr','Other'],inplace=True)


In [ ]:

mapping = {
    "Mr":0,
    "Miss":1,
    "Mrs" : 1,
    "Master":2,
    "Other":3
}



mapping_sex = {
    'male' : 0,
    'female': 1
}

mapping_em = {
    'S' :0,
    'C' :1,
    'Q' :2
}


In [ ]:

data['Initial'] = data['Initial'].map(mapping)
data['Fare'] = data['Fare'].map(lambda x : np.log(x) if x > 0 else 0)
data['Sex'] = data['Sex'].map(mapping_sex)
data['Embarked'] = data['Embarked'].map(mapping_em)


In [ ]:
data.drop(['PassengerId', "Ticket", "Cabin", "Name"], axis = 1, inplace = True)

data.loc[ (data['Age'].isnull()) & (data['Initial'] == 0), 'Age' ] = 32
data.loc[ (data['Age'].isnull()) & (data['Initial'] == 1), 'Age' ] = 28
data.loc[ (data['Age'].isnull()) & (data['Initial'] == 2), 'Age' ] = 5
data.loc[ (data['Age'].isnull()) & (data['Initial'] == 3), 'Age' ] = 45

In [ ]:
y = data['Survived']
X = data.drop('Survived', axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=10)

In [ ]:
xgb = XGBClassifier()

xgb_param_grid = {
    'n_estimators' : [100, 200, 400, 600],
    'learning_rate' : [0.01, 0.05, 0.1, 0.15, 0.2],
    'max_depth' : [4, 6, 8, 10, 12],
}

xgb_grid = GridSearchCV(xgb, param_grid = xgb_param_grid, scoring="accuracy", n_jobs= -1, verbose = 1)
xgb_grid.fit(X_train, y_train)


In [ ]:
print("?? ?? ??? : {0:.4f}".format(xgb_grid.best_score_))
print("??? ???? : ", xgb_grid.best_params_)

In [ ]:
result = pd.DataFrame(xgb_grid.cv_results_)
result.sort_values(by=['rank_test_score'], inplace=True)

In [ ]:
result[['params', 'mean_test_score', 'rank_test_score']].head(10)

In [ ]:
xgb = XGBClassifier(n_estimators=400, learning_rate = 0.1, max_depth = 3)
evals = [(X_test, y_test)]
xgb.fit(X_train, y_train, early_stopping_rounds = 100, eval_metric = "logloss", eval_set = evals, verbose = 1)

In [ ]:
xgb = XGBClassifier(n_estimators=400, learning_rate = 0.1, max_depth = 3)
evals = [(X_test, y_test)]
xgb.fit(X_train, y_train, early_stopping_rounds = 3, eval_metric = "logloss", eval_set = evals, verbose = 1)

In [ ]:
fig, ax = plt.subplots()
plot_importance(xgb, ax=ax)

In [ ]:
from lightgbm import LGBMClassifier, plot_importance

In [ ]:
lgb = LGBMClassifier(n_estimaotrs = 400)
lgb.fit(X_train, y_train)

In [ ]:
lgb_pred = lgb.predict(X_test)
metrics(y_test, lgb_pred)

In [ ]:
lgb = LGBMClassifier(n_estimaotrs = 400)
evals = [(X_test, y_test)]
lgb.fit(X_train, y_train, early_stopping_rounds = 100, eval_metric = "logloss", eval_set = evals, verbose = True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_importance(lgb, ax = ax)